# 06. Sliding Window: Advanced Problems

The same expand/shrink skeleton in trickier clothes: **window of the complement**, **bonus windows**, **delete one**, **exactly K**, **sort + window**, **reducing to zero**.

**Learning order:** MAX POINTS FROM CARDS → GRUMPY OWNER → DELETE ONE ELEMENT → K DIFFERENT INTEGERS → MOST FREQUENT ELEMENT → REDUCE X TO ZERO → INTERVIEW THINKING

## 1. Maximum Points You Can Obtain from Cards

Take `k` cards from the **ends** of the row. Taking from the ends means the **untaken cards form one contiguous window** of size `n - k`. Maximize the taken sum = `total - minimum window sum` of size `n - k`.

**Lesson:** if the chosen items are at the ends, think about the **middle** as a window.

In [1]:
def max_score(card_points, k):
    n, window = len(card_points), len(card_points) - k
    cur = sum(card_points[:window])
    best = cur
    for i in range(window, n):
        cur += card_points[i] - card_points[i - window]
        best = min(best, cur)
    return sum(card_points) - best

print(max_score([1, 2, 3, 4, 5, 6, 1], 3), max_score([2, 2, 2], 2))   # 12 4

12 4


## 2. Grumpy Bookstore Owner (fixed window of "bonus")

Customers are always happy when the owner is not grumpy. With a secret technique the owner can stay calm for `minutes` consecutive minutes. Baseline = customers already satisfied. The window picks the stretch where **calming the owner gains the most** (sum of customers during grumpy minutes inside the window).

In [2]:
def max_satisfied(customers, grumpy, minutes):
    base = sum(c for c, g in zip(customers, grumpy) if not g)
    extra = best = 0
    for i, (c, g) in enumerate(zip(customers, grumpy)):
        if g:
            extra += c
        if i >= minutes and grumpy[i - minutes]:
            extra -= customers[i - minutes]
        best = max(best, extra)
    return base + best

print(max_satisfied([1, 0, 1, 2, 1, 1, 7, 5], [0, 1, 0, 1, 0, 1, 0, 1], 3))   # 16

16


## 3. Longest Subarray of 1's After Deleting One Element

Deleting one element means we are allowed **at most one zero** inside the window, and the deleted element itself does not count in the length: `length = right - left` (window size minus 1). If the array has no zeros you must still delete one element.

In [3]:
def longest_subarray_after_delete(nums):
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1:
            zeros -= nums[left] == 0
            left += 1
        best = max(best, right - left)             # window length minus the one deleted
    return best

print(longest_subarray_after_delete([1, 1, 0, 1]), longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]), longest_subarray_after_delete([1, 1, 1]))   # 3 5 2

3 5 2


## 4. Subarrays with K Different Integers (exactly K)

"Exactly K" is not monotonic, so use **`atMost(K) - atMost(K - 1)`**, where `atMost(k)` counts subarrays with at most `k` distinct values (`right - left + 1` per step).

In [4]:
from collections import defaultdict

def subarrays_with_k_distinct(nums, k):
    def at_most(k):
        count, left, res = defaultdict(int), 0, 0
        for right, x in enumerate(nums):
            count[x] += 1
            while len(count) > k:
                count[nums[left]] -= 1
                if count[nums[left]] == 0:
                    del count[nums[left]]
                left += 1
            res += right - left + 1
        return res
    return at_most(k) - at_most(k - 1)

print(subarrays_with_k_distinct([1, 2, 1, 2, 3], 2), subarrays_with_k_distinct([1, 2, 1, 3, 4], 3))   # 7 3

7 3


## 5. Frequency of the Most Frequent Element (sort + window)

In one operation you may increment an element by 1. After at most `k` operations, what is the highest frequency? **Sort**, then for a window `[left, right]` we raise every element to `nums[right]`. Cost = `nums[right] * window_size - window_sum`. Shrink while the cost exceeds `k`.

In [5]:
def max_frequency(nums, k):
    nums.sort()
    left = total = best = 0
    for right, x in enumerate(nums):
        total += x
        while x * (right - left + 1) - total > k:
            total -= nums[left]
            left += 1
        best = max(best, right - left + 1)
    return best

print(max_frequency([1, 2, 4], 5), max_frequency([1, 4, 8, 13], 5), max_frequency([3, 9, 6], 2))   # 3 2 1

3 2 1


## 6. Minimum Operations to Reduce X to Zero

Remove numbers from the **ends** to get sum exactly `x`, using the fewest removals. Equivalent: the **remaining middle** has sum `total - x`. Find the **longest** subarray with that sum (positive numbers → sliding window). Answer = `n - longest`.

In [6]:
def min_operations(nums, x):
    target = sum(nums) - x
    if target < 0:
        return -1
    left = cur = 0
    best = -1
    for right, v in enumerate(nums):
        cur += v
        while cur > target:
            cur -= nums[left]
            left += 1
        if cur == target:
            best = max(best, right - left + 1)
    return -1 if best == -1 else len(nums) - best

print(min_operations([1, 1, 4, 2, 3], 5), min_operations([5, 6, 7, 8, 9], 4), min_operations([3, 2, 20, 1, 1, 3], 10))   # 2 -1 5

2 -1 5


## 7. Interview thinking

**Transform the problem first:** items at the **ends** → look at the **middle** window; "delete one" → allow one bad element; "exactly K" → at most K minus at most K-1; "make equal with increments" → sort + cost formula.

**Say:** "I can reformulate this as finding the longest/shortest window with property ...; then it is a standard sliding window in O(n)."

**Common mistakes:** forgetting the `- 1` for the deleted element, exactly-K without the difference trick, sliding window on arrays with negatives (use prefix sums), not sorting in the frequency problem, `target < 0` edge case.

## Quick Revision

- Ends → middle window (`n - k`). Max taken = `total - min middle`.
- Delete one = at most one zero, length `right - left`.
- Exactly K = atMost(K) − atMost(K−1).
- Frequency: sort, cost `x * size - sum ≤ k`.
- Reduce to zero: longest subarray with sum `total - x`.